In [1]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
!pip install -q langchain-text-splitters

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    "Machine learning enables systems to learn from data without explicit programming.",
    "Deep learning uses neural networks with many layers for complex pattern recognition.",
    "RAG combines retrieval systems with LLMs to generate context-aware answers.",
    "Vector databases store embeddings for semantic search."
]

print("Documents loaded:", len(documents))

Documents loaded: 4


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = []
for doc in documents:
    chunks.extend(text_splitter.split_text(doc))

print("Total chunks:", len(chunks))

Total chunks: 4


In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
embeddings = model.encode(chunks)

embeddings = np.array(embeddings).astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (4, 384)


In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index built ✅")

FAISS index built ✅


In [10]:
import numpy as np

In [11]:
def retrieve(query, k=3):
    # convert query → embedding
    query_vec = model.encode([query]).astype("float32")

    # search nearest neighbors
    distances, indices = index.search(query_vec, k)

    # return top matching chunks
    results = [chunks[i] for i in indices[0]]

    return results

In [12]:
query = "What is retrieval augmented generation?"

results = retrieve(query)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n{r}")


Result 1:
RAG combines retrieval systems with LLMs to generate context-aware answers.

Result 2:
Vector databases store embeddings for semantic search.

Result 3:
Machine learning enables systems to learn from data without explicit programming.


In [13]:
def generate_answer(query):
    # Step 1: retrieve relevant context
    context = retrieve(query, k=3)

    # Step 2: build prompt
    prompt = f"""
You are a helpful AI assistant.

Use the context below to answer the question.

Context:
{chr(10).join(context)}

Question:
{query}

Answer clearly and concisely:
"""

    return prompt

In [14]:
query = "What is RAG in machine learning?"

print(generate_answer(query))


You are a helpful AI assistant.

Use the context below to answer the question.

Context:
RAG combines retrieval systems with LLMs to generate context-aware answers.
Machine learning enables systems to learn from data without explicit programming.
Deep learning uses neural networks with many layers for complex pattern recognition.

Question:
What is RAG in machine learning?

Answer clearly and concisely:



In [15]:
chat_history = []

def chat(query):
    # retrieve context
    context = retrieve(query, k=3)

    # build prompt with memory
    history_text = "\n".join(chat_history[-4:])  # last 2 exchanges

    prompt = f"""
You are a helpful AI assistant.

Conversation history:
{history_text}

Context:
{chr(10).join(context)}

User: {query}
Assistant:
"""

    # store conversation
    chat_history.append(f"User: {query}")
    chat_history.append(f"Assistant: (generated using context)")

    return prompt

In [16]:
print(chat("What is RAG?"))
print(chat("Why is it useful in AI systems?"))


You are a helpful AI assistant.

Conversation history:


Context:
RAG combines retrieval systems with LLMs to generate context-aware answers.
Vector databases store embeddings for semantic search.
Machine learning enables systems to learn from data without explicit programming.

User: What is RAG?
Assistant:


You are a helpful AI assistant.

Conversation history:
User: What is RAG?
Assistant: (generated using context)

Context:
Machine learning enables systems to learn from data without explicit programming.
Deep learning uses neural networks with many layers for complex pattern recognition.
Vector databases store embeddings for semantic search.

User: Why is it useful in AI systems?
Assistant:



In [17]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 54.0 MB/s eta 0:00:00


In [9]:
%%writefile app.py

import streamlit as st
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# -----------------------------
# Sample knowledge base
# -----------------------------
documents = [
    "Machine learning enables systems to learn from data.",
    "Deep learning uses neural networks with many layers.",
    "RAG combines retrieval with language models.",
    "Vector databases store embeddings for similarity search."
]

# -----------------------------
# Load embedding model
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# Build FAISS index
# -----------------------------
chunks = documents
embeddings = model.encode(chunks).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# -----------------------------
# Retrieval function
# -----------------------------
def retrieve(query, k=2):
    q_emb = model.encode([query]).astype("float32")
    _, idx = index.search(q_emb, k)
    return [chunks[i] for i in idx[0]]

# -----------------------------
# Streamlit UI
# -----------------------------
st.title("RAG Chatbot")

query = st.text_input("Ask a question:")

if query:
    context = retrieve(query)

    st.subheader("Retrieved Context")
    for c in context:
        st.write("-", c)

    st.subheader("Answer (Simulated LLM)")
    st.write("Based on retrieved context:", context[0])

Writing app.py


In [10]:
!pip install -q streamlit

In [11]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.179.60:8501

  Stopping...
  Stopping...
